# **Concert Travel Agent**

An agent that recommends upcoming concerts for the given artist providing
real time data for concert tickts, flights, but also recommends similar artists with upcoming concerts if the original one does not have any.

## Run each cell before running the agent

In [ ]:
!pip install -q openai pandas requests numpy

import openai
import pandas as pd
import numpy as np
import requests
import json
import time
import os
from datetime import datetime, timedelta
from typing import List, Dict, Optional, Tuple
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
client = openai.OpenAI(api_key=OPENAI_API_KEY)

TICKETMASTER_API_KEY = userdata.get('TICKETMASTER_API_KEY')
SETLIST_API_KEY = userdata.get('SETLIST_API_KEY')
SERPAPI_KEY = userdata.get('SERPAPI_KEY')
LASTFM_KEY = userdata.get('LASTFM_KEY')


Build similar artist database using Last.fm

- saves the results in files that can be accessed and used
- approx. time of execution: 5 mins

In [ ]:
BASE_URL = "http://ws.audioscrobbler.com/2.0/"

SEED_ARTISTS = [
    "Taylor Swift", "Drake", "Arctic Monkeys", "Billie Eilish",
    "Phoebe Bridgers", "The Weeknd", "Kendrick Lamar", "BTS",
    "Olivia Rodrigo", "The Script"
]

def get_similar_artists(artist, limit=10):
    params = {
        "method": "artist.getsimilar",
        "artist": artist,
        "api_key": LASTFM_KEY,
        "format": "json",
        "limit": limit
    }
    res = requests.get(BASE_URL, params=params)
    data = res.json()
    return [a["name"] for a in data.get("similarartists", {}).get("artist", [])]

#genre - 5 types
def get_tags(artist):
    params = {
        "method": "artist.gettoptags",
        "artist": artist,
        "api_key": LASTFM_KEY,
        "format": "json"
    }
    res = requests.get(BASE_URL, params=params)
    data = res.json()
    return [t["name"] for t in data.get("toptags", {}).get("tag", [])[:5]]

def build_dataset_expanded(depth=2, similar_limit=10):
    visited = set()
    queue = list(SEED_ARTISTS)
    dataset = []

    for _ in range(depth):
        next_queue = []
        for artist in queue:
            if artist in visited:
                continue
            visited.add(artist)
            print(f"Fetching {artist}...")
            try:
                similar = get_similar_artists(artist, limit=similar_limit)
                tags = get_tags(artist)

                record = {
                    "artist": artist,
                    "genres": tags,
                    "similar_artists": similar
                }

                dataset.append(record)
                next_queue.extend(similar)
                time.sleep(1)
            except Exception as e:
                print(f"Error with {artist}: {e}")

        queue = next_queue

    with open("artists_dataset.json", "w") as f:
        json.dump(dataset, f, indent=2)
    print(f"Dataset saved: {len(dataset)} artists")

# build_dataset_expanded()

In [ ]:
# from google.colab import files

# files.download("artists_dataset.json")

In [ ]:
def setup_rag(json_path="artists_dataset.json"):

    if not os.path.exists("/content/artists_dataset.json"):
      !gdown 1ZmSFulmoQS-FilNcqxMnrm59pDknzT1K -O artists_dataset.json

    vs = client.vector_stores.create(name="artists-rag-store")

    client.vector_stores.files.upload_and_poll(
        vector_store_id=vs.id,
        file=open(json_path, "rb")
    )

    return vs.id
VECTOR_STORE_ID = setup_rag()

In [ ]:
with open("artists_dataset.json") as f:
    _artist_data = {a["artist"].lower(): a for a in json.load(f)}

In [ ]:
NORTH_AMERICA_CODES = {'US', 'CA', 'MX'}
EUROPEAN_COUNTRY_CODES = {
    'GB','FR','DE','ES','IT','NL','BE','CH','AT','IE',
    'SE','DK','NO','FI','PL','PT'
}

#obtain unique artist id to then fetch upcoming events
def get_attraction_id(artist_name: str) -> str | None:
    params = {
        'apikey': TICKETMASTER_API_KEY,
        'keyword': artist_name,
        'classificationName': 'music',
        'size': 5
    }
    response = requests.get(
        "https://app.ticketmaster.com/discovery/v2/attractions.json",
        params=params, timeout=10
    )
    data = response.json()
    attractions = data.get('_embedded', {}).get('attractions', [])

    if not attractions:
        return None

    for attraction in attractions:
        if attraction.get('name', '').lower() == artist_name.lower():
            return attraction['id']
    return attractions[0]['id']


#fetch upcoming events of an artist based onn their id
def search_artist_concerts(artist_name: str) -> List[Dict]:
    attraction_id = get_attraction_id(artist_name)
    if not attraction_id:
        return []

    params = {
        'apikey': TICKETMASTER_API_KEY,
        'attractionId': attraction_id,
        'classificationName': 'music',
        'size': 50,
        'sort': 'date,asc'
    }

    all_concerts = []
    try:
        response = requests.get(
            "https://app.ticketmaster.com/discovery/v2/events.json",
            params=params,
            timeout=15
        )
        data = response.json()
        events = data.get('_embedded', {}).get('events', [])

        for event in events:
            try:
                venue = event['_embedded']['venues'][0]
                country_code = venue.get('country', {}).get('countryCode', '')

                if country_code in EUROPEAN_COUNTRY_CODES:
                    region = 'europe'
                elif country_code in NORTH_AMERICA_CODES:
                    region = 'north_america'
                else:
                    region = 'international'

                all_concerts.append({
                    'id': event['id'],
                    'artist': artist_name,
                    'venue': venue['name'],
                    'city': venue.get('city', {}).get('name', 'Unknown'),
                    'country': venue.get('country', {}).get('name', 'Unknown'),
                    'country_code': country_code,
                    'date': event['dates']['start']['localDate'],
                    'url': event.get('url', ''),
                    'region': region,
                })
            except (KeyError, IndexError):
                continue

    except Exception as e:
        print(f"Ticketmaster error: {e}")

    eu_concerts = sum(1 for c in all_concerts if c['region'] == 'europe')
    print(f"Found {len(all_concerts)} concerts - {eu_concerts} in Europe")
    return all_concerts


In [ ]:
SETLIST_API_BASE = "https://api.setlist.fm/rest/1.0"
MUSICBRAINZ_URL = "https://musicbrainz.org/ws/2/artist"

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def make_session_with_retries():
    session = requests.Session()
    retry = Retry(total=3, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session

_session = make_session_with_retries()
_setlist_cache = {}
_mbid_cache = {}

KPOP_SOLO = {
    "lisa": "KR", "jennie": "KR", "jisoo": "KR",
    "rosé": "KR", "rose": "KR", "jimin": "KR",
    "v": "KR", "suga": "KR", "j-hope": "KR", "rm": "KR",
}

def get_best_mbid(artist_name: str, genres: List[str] = None) -> str | None:
    cache_key = artist_name.lower().strip()
    if cache_key in _mbid_cache:
        return _mbid_cache[cache_key]

    headers = {"User-Agent": "my-music-app/1.0 (youremail@example.com)"}
    try:
        resp = _session.get(
            MUSICBRAINZ_URL,
            params={"query": artist_name, "fmt": "json", "limit": 10},
            headers=headers, timeout=15
        )
        resp.raise_for_status()
    except requests.exceptions.SSLError:
        try:
            resp = requests.get(
                MUSICBRAINZ_URL,
                params={"query": artist_name, "fmt": "json", "limit": 10},
                headers=headers, timeout=15
            )
            resp.raise_for_status()
        except Exception as e:
            print(f"MusicBrainz fallback failed: {e}")
            return None
    except Exception as e:
        print(f"MusicBrainz error: {e}")
        return None

    artists = resp.json().get("artists", [])
    if not artists:
        return None

    def score(a):
        s = int(a.get("score", 0))
        name = a.get("name", "").lower()
        query = artist_name.lower()
        if name == query: s += 100
        elif query in name: s += 30
        if genres:
            genre_str = " ".join(genres).lower()
            if "k-pop" in genre_str or "korean" in genre_str:
                if a.get("country") == "KR": s += 50
        if query in KPOP_SOLO:
            if a.get("country") == "KR": s += 80
        if a.get("type") in ("Person", "Group"): s += 10
        return s

    result = max(artists, key=score).get("id")
    _mbid_cache[cache_key] = result
    return result


def parse_setlist(valid_setlist: Dict, artist_name: str) -> Dict:
    songs = []
    for set_part in valid_setlist.get("sets", {}).get("set", []):
        for song in set_part.get("song", []):
            try:
                name = song.get("name")
                if isinstance(name, dict):
                    name = name.get("text") or name.get("value") or str(name)
                if name and isinstance(name, str) and name.strip():
                    songs.append(name.strip())
            except Exception:
                continue

    venue_data = valid_setlist.get("venue", {})
    venue_name = venue_data.get("name") if isinstance(venue_data, dict) else str(venue_data)
    artist_data = valid_setlist.get("artist", {})
    artist_out = (artist_data.get("name") if isinstance(artist_data, dict) else str(artist_data)) or artist_name

    return {
        "songs": songs,
        "date": valid_setlist.get("eventDate"),
        "venue": venue_name or "Unknown",
        "artist": artist_out
    }


def search_artist_setlist(artist_name: str) -> Dict:
    cache_key = artist_name.lower().strip()
    if cache_key in _setlist_cache:
        print(f"Setlist cached for {artist_name}")
        return _setlist_cache[cache_key]

    headers = {
        "x-api-key": SETLIST_API_KEY,
        "Accept": "application/json",
        "User-Agent": "my-music-app/1.0 (youremail@example.com)"
    }

    try:
        artist_genres = _artist_data.get(cache_key, {}).get("genres", [])
        mbid = get_best_mbid(artist_name, genres=artist_genres)
        time.sleep(1)

        if not mbid:
            result = {"songs": [], "error": "Could not resolve MusicBrainz ID"}
            _setlist_cache[cache_key] = result
            return result

        setlist_resp = _session.get(
            f"{SETLIST_API_BASE}/artist/{mbid}/setlists",
            headers=headers, params={"p": 1}, timeout=10
        )

        if setlist_resp.status_code == 429:
            print(f"Setlist.fm rate limited — waiting 10s...")
            time.sleep(10)
            setlist_resp = _session.get(
                f"{SETLIST_API_BASE}/artist/{mbid}/setlists",
                headers=headers, params={"p": 1}, timeout=10
            )

        setlists = setlist_resp.json().get("setlist", [])
        if not setlists:
            result = {"songs": [], "error": "No setlists found"}
            _setlist_cache[cache_key] = result
            return result

        valid_setlist = None
        for s in setlists:
            sets = s.get("sets", {}).get("set", [])
            has_songs = any(
                song.get("name")
                for set_part in sets
                for song in set_part.get("song", [])
            )
            if has_songs:
                valid_setlist = s
                break

        if not valid_setlist:
            valid_setlist = setlists[0]

        result = parse_setlist(valid_setlist, artist_name)
        _setlist_cache[cache_key] = result
        return result

    except Exception as e:
        print(f"Setlist error: {e}")
        result = {"songs": [], "error": str(e)}
        _setlist_cache[cache_key] = result
        return result


In [ ]:
import re
def normalise_song_name(name) -> str:
    if not isinstance(name, str):
        name = str(name)

    name = name.lower().strip()
    name = re.sub(r"[\-\–\—\_\.\,\!\?\'\"\(\)]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name


def check_setlist(artist_name: str, songs_to_check: List[str] = None) -> Dict:
    setlist_data = search_artist_setlist(artist_name)
    if not setlist_data.get("songs"):
        return {"error": f"No setlist found for {artist_name}"}

    result = {
        "setlist_songs": setlist_data["songs"],
        "last_played": setlist_data.get("date"),
        "venue": setlist_data.get("venue")
    }

    if songs_to_check:
        normalised_setlist = [normalise_song_name(s) for s in setlist_data["songs"]]
        in_setlist = [
            s for s in songs_to_check
            if normalise_song_name(s) in normalised_setlist
        ]
        result["will_be_played"] = in_setlist
        result["not_in_setlist"] = [s for s in songs_to_check if s not in in_setlist]

    return result


In [ ]:
AIRPORT_CODES = {
    'dublin': 'DUB', 'cork': 'ORK', 'shannon': 'SNN', 'belfast': 'BFS',
    'london': 'LHR', 'paris': 'CDG', 'madrid': 'MAD', 'barcelona': 'BCN',
    'brussels': 'BRU', 'amsterdam': 'AMS', 'munich': 'MUC', 'berlin': 'BER',
    'frankfurt': 'FRA', 'rome': 'FCO', 'milan': 'MXP', 'vienna': 'VIE',
    'zurich': 'ZRH', 'geneva': 'GVA', 'copenhagen': 'CPH', 'stockholm': 'ARN',
    'oslo': 'OSL', 'helsinki': 'HEL', 'warsaw': 'WAW', 'prague': 'PRG',
    'budapest': 'BUD', 'athens': 'ATH', 'istanbul': 'IST', 'lisbon': 'LIS',
    'porto': 'OPO', 'new york': 'JFK', 'los angeles': 'LAX', 'chicago': 'ORD',
    'miami': 'MIA', 'las vegas': 'LAS', 'boston': 'BOS', 'seattle': 'SEA',
    'san francisco': 'SFO', 'toronto': 'YYZ', 'tokyo': 'NRT', 'seoul': 'ICN',
    'sydney': 'SYD', 'manchester': 'MAN', 'birmingham': 'BHX', 'edinburgh': 'EDI',
    'glasgow': 'GLA', 'napa': 'SFO', 'auckland': 'AKL', 'melbourne': 'MEL',
    'vancouver': 'YVR', 'montreal': 'YUL', 'luxembourg': 'LUX',
    'bratislava': 'BTS', 'zagreb': 'ZAG', 'riga': 'RIX', 'vilnius': 'VNO',
    'tallinn': 'TLL', 'reykjavik': 'KEF', 'belgrade': 'BEG', 'bucharest': 'OTP',
    'sofia': 'SOF', 'nice': 'NCE', 'lyon': 'LYS', 'seville': 'SVQ',
    'valencia': 'VLC', 'bilbao': 'BIO', 'hamburg': 'HAM', 'cologne': 'CGN',
    'dusseldorf': 'DUS', 'stuttgart': 'STR', 'bologna': 'BLQ', 'naples': 'NAP',
    'florence': 'FLR', 'gothenburg': 'GOT', 'malmo': 'MMX', 'aarhus': 'AAR',
    'bergen': 'BGO', 'krakow': 'KRK', 'wroclaw': 'WRO',
    'hartford': 'BDL', 'providence': 'PVD', 'buffalo': 'BUF', 'cleveland': 'CLE',
    'detroit': 'DTW', 'minneapolis': 'MSP', 'denver': 'DEN', 'phoenix': 'PHX',
    'atlanta': 'ATL', 'dallas': 'DFW', 'houston': 'IAH', 'nashville': 'BNA',
    'charlotte': 'CLT', 'philadelphia': 'PHL', 'washington': 'DCA',
    'baltimore': 'BWI', 'orlando': 'MCO', 'tampa': 'TPA', 'san diego': 'SAN',
    'portland': 'PDX', 'salt lake city': 'SLC', 'kansas city': 'MCI',
    'st. louis': 'STL', 'pittsburgh': 'PIT', 'indianapolis': 'IND',
    'columbus': 'CMH', 'milwaukee': 'MKE', 'new orleans': 'MSY',
    'memphis': 'MEM', 'raleigh': 'RDU', 'sacramento': 'SMF', 'san jose': 'SJC',
    'queens': 'JFK', 'brooklyn': 'JFK', 'bronx': 'JFK', 'newark': 'EWR',
    'inglewood': 'LAX', 'east rutherford': 'EWR', 'foxborough': 'BOS',
    'paradise': 'LAS', 'sunrise': 'FLL', 'elmont': 'JFK',
}

def city_to_airport_code(city_name: str):
    city_lower = city_name.lower().strip()
    for city, code in AIRPORT_CODES.items():
      #cehck exact matching + partial matching
        if city == city_lower or city in city_lower or city_lower in city:
            return code
    #best assumption based on city name
    clean = ''.join(c for c in city_name if c.isalpha())
    return clean[:3].upper() if len(clean) >= 3 else None


def get_user_region(origin_city: str) -> str:
    city_lower = origin_city.lower().strip()
    north_america = {
        'chicago','new york','los angeles','miami','boston','seattle',
        'san francisco','las vegas','toronto','vancouver','montreal'
    }
    europe = {
        'london','paris','berlin','madrid','rome','amsterdam','dublin',
        'brussels','vienna','zurich','stockholm','oslo','copenhagen',
        'helsinki','warsaw','lisbon','bucharest'
    }
    if city_lower in north_america: return 'north_america'
    if city_lower in europe: return 'europe'
    return 'international'


SERPAPI_ENDPOINT = "https://serpapi.com/search.json"

def get_flight_info(api_key, departure, arrival, outbound_date, return_date=None):
    params = {
        "engine": "google_flights",
        "departure_id": departure,
        "arrival_id": arrival,
        "outbound_date": outbound_date,
        "currency": "USD",
        "hl": "en",
        "api_key": api_key,
        "sort_by": 1
    }
    #round-trip or one-way
    if return_date:
        params["return_date"] = return_date
        params["type"] = 1
    else:
        params["type"] = 2

    try:
        response = requests.get(SERPAPI_ENDPOINT, params=params, timeout=15)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        # print(f"Flight API error: {e}")
        return None

    flights = data.get("best_flights", []) + data.get("other_flights", [])
    if not flights:
        return None

    cheapest = flights[0]

    return {
        "price": cheapest["price"],
        "currency": "USD",
        "airline": cheapest["flights"][0].get("airline", "Various"),
        "total_duration_minutes": cheapest["total_duration"],
        "booking_link": data.get("search_metadata", {}).get("google_flights_url"),
        "price_level": data.get("price_insights", {}).get("price_level"),
    }


def get_flight_info_with_retry(api_key, departure, arrival, outbound_date, return_date=None, retries=2):
    for attempt in range(retries + 1):
        try:
            result = get_flight_info(api_key, departure, arrival, outbound_date, return_date)
            return result
        except Exception as e:
            if attempt < retries:
                wait = 3 * (attempt + 1)
                print(f"SerpAPI timeout - retrying in {wait}s...")
                time.sleep(wait)
            else:
                print(f"Flight API failed after {retries} retries: {e}")
                return None


def get_flight_price(origin_city: str, destination_city: str, outbound_date: str, return_date: str = None) -> Dict:
    origin_code = city_to_airport_code(origin_city)
    dest_code = city_to_airport_code(destination_city)

    if not origin_code or not dest_code:
        return {"error": f"Could not resolve airport for {origin_city} or {destination_city}"}

    concert_date = datetime.strptime(outbound_date, "%Y-%m-%d")
    outbound = (concert_date - timedelta(days=1)).strftime("%Y-%m-%d")
    ret = return_date or (concert_date + timedelta(days=1)).strftime("%Y-%m-%d")

    oneway_info = get_flight_info_with_retry(SERPAPI_KEY, origin_code, dest_code, outbound)
    time.sleep(1)
    roundtrip_info = get_flight_info_with_retry(SERPAPI_KEY, origin_code, dest_code, outbound, ret)

    if not oneway_info and not roundtrip_info:
        return {"error": f"No flights found from {origin_city} to {destination_city}"}

    best = roundtrip_info or oneway_info
    return {
        "origin": origin_city,
        "destination": destination_city,
        "outbound_date": outbound,
        "return_date": ret,
        "flight_price_oneway": oneway_info["price"] if oneway_info else None,
        "flight_price_roundtrip": roundtrip_info["price"] if roundtrip_info else None,
        "currency": best["currency"],
        "airline": best["airline"],
        "duration_mins": best["total_duration_minutes"],
        "booking_link": best["booking_link"],
        "trip_type": "round trip" if ret else "one way",
    }


In [ ]:
def analyze_single_concert(concert, origin_city, origin_code, region_label, serpapi_key):

    if origin_city.lower() == concert["city"].lower():
        print(f"Local concert!! no flight needed")
        return {
            "concert": concert,
            "flight_price_oneway": 0,
            "flight_price_roundtrip": 0,
            "flight_currency": "N/A",
            "flight_link": None,
            "airline": None,
            "duration_mins": 0,
            "is_local": True,
            "region": region_label,
        }

    dest_code = city_to_airport_code(concert["city"])
    if not dest_code:
        print(f"  Skipping {concert['city']} — no airport code")
        return None

    concert_date = datetime.strptime(concert["date"], "%Y-%m-%d")
    outbound_date = (concert_date - timedelta(days=1)).strftime("%Y-%m-%d")
    return_date = (concert_date + timedelta(days=1)).strftime("%Y-%m-%d")

    # print(f"Searching {origin_code} → {dest_code} | out {outbound_date} | return {return_date}")

    oneway_info = get_flight_info_with_retry(serpapi_key, origin_code, dest_code, outbound_date)
    time.sleep(1)
    roundtrip_info = get_flight_info_with_retry(serpapi_key, origin_code, dest_code, outbound_date, return_date)

    if not oneway_info and not roundtrip_info:
        print(f"  No flights found for {concert['city']}")
        return None

    best = roundtrip_info or oneway_info
    return {
        "concert": concert,
        "flight_price_oneway":    oneway_info["price"]    if oneway_info    else None,
        "flight_price_roundtrip": roundtrip_info["price"] if roundtrip_info else None,
        "flight_currency": "USD",
        "flight_link": best["booking_link"],
        "airline": best.get("airline", "Various"),
        "duration_mins": (oneway_info or roundtrip_info).get("total_duration_minutes", 0),
        "is_local": False,
        "region": region_label,
    }

In [ ]:
def analyze_concerts_for_user(artist_name: str, origin_city: str) -> Dict:
    print(f"\n Analyzing concerts for {artist_name}")

    concerts = search_artist_concerts(artist_name)

    if not concerts:
      return
        # print(f"No concerts found")
        # return find_concerts_for_similar_artists(artist_name, origin_city)

    origin_code = city_to_airport_code(origin_city)
    user_region = get_user_region(origin_city)

    region_configs = (
        [("north_america", 5), ("europe", 3)]
        if user_region == "north_america"
        else [("europe", 5), ("international", 3)]
    )

    analyzed = []
    for region, limit in region_configs:
        filtered = [c for c in concerts if c.get("region") == region]
        for concert in filtered[:limit]:
            print("Looking for flights...")
            result = analyze_single_concert(
                concert=concert,
                origin_city=origin_city,
                origin_code=origin_code,
                region_label=region,
                serpapi_key=SERPAPI_KEY
            )
            if result:
                analyzed.append(result)

    if not analyzed:
        return {"has_concerts": True, "error": "No flight data available", "concerts": concerts}

    #rank: local first, then cheapest round trip
    analyzed.sort(key=lambda x: (
        0 if x.get("is_local") else 1,
        x.get("flight_price_roundtrip") or x.get("flight_price_oneway") or float("inf")
    ))

    final_result = {
        "has_concerts": True,
        "artist": artist_name,
        "concerts_analyzed": analyzed,
        "best_concert": analyzed[0],
    }

    return final_result


In [ ]:
SETLIST_TRIGGERS = ["setlist", "songs they play", "what songs", "what do they play", "tracklist"]

def looks_like_concert_query(message: str) -> bool:
    msg_lower = message.lower()

    #if it's a setlist question, don't force analyze_concerts_for_user - should check the setlist instead
    if any(t in msg_lower for t in SETLIST_TRIGGERS):
        return False

    concert_triggers = ["concert", "tour", "show", "tickets", "see live", "what about", "gig", "concerts", "shows"]
    return any(t in msg_lower for t in concert_triggers)

In [ ]:
def find_similar_artists_touring(original_artist: str, origin_city: str) -> Dict:
    similar = get_similar_artists(original_artist, limit=4)
    touring = []
    for artist in similar:
        concerts = search_artist_concerts(artist)
        if concerts:
            eu = sum(1 for c in concerts if c.get("region") == "europe")
            na = sum(1 for c in concerts if c.get("region") == "north_america")
            touring.append({
                "artist": artist,
                "total_concerts": len(concerts),
                "european_concerts": eu,
                "north_american_concerts": na,
                "next_concert": concerts[0]
            })
        if len(touring) >= 5:
            break
    return {"original_artist": original_artist, "similar_artists_touring": touring}


def run_concert_travel_agent():
    print("\n" + "-" * 40)
    print("CONCERT TRAVEL ASSISTANT")
    print("-" * 40)
    print("""
I am here to help you decide whether to fly to see your favourite artists!

Tell me:
  • Who you want to see
  • Where you're flying from

Examples:
  "I love The Weeknd, I'm in Dublin"
  "BTS concerts, I'm in Berlin"
  "Will they play Tattoo at the Lorde concert?"
  "Who else is touring right now?" - for similar artist recommendation

Type 'quit' to exit
""")
    print("-" * 50)

    SYSTEM_PROMPT = """You are a Concert Travel Assistant - friendly, conversational, and honest.

CRITICAL: NEVER answer questions about concerts, setlists, or flights from your own knowledge.
ALWAYS call the appropriate tool first. Your training data is outdated - only the tools have current data.
When a user mentions an artist and a city - call analyze_concerts_for_user immediately.

ABSOLUTE RULE: The FIRST tool call for ANY artist + city combination MUST be
analyze_concerts_for_user. No exceptions. Not file_search. Not find_similar_artists_touring.
analyze_concerts_for_user FIRST, every single time, no matter what you think you know.

If the message contains '[User's city from earlier in conversation: X]',
use X as the origin_city when calling any tool that requires it.
NEVER call find_similar_artists_touring as a first response — always call
analyze_concerts_for_user first to check if the artist actually has concerts.
find_similar_artists_touring is only for when analyze_concerts_for_user
returns has_concerts: False.

TOOLS AND WHEN TO USE THEM:

1. analyze_concerts_for_user
   - ONLY if the user requests a NEW artist or NEW city never searched before e.g "what about *insert new artist name* concerts?". DO NOT call find_simialr_artists_touring or use file_search for this
   - Fetches real concerts from Ticketmaster and real flights from Google Flights
   - Ranks results by cheapest round-trip flight first
   - Do NOT call this if you already have results for this artist in the conversation

2. check_setlist
   - User asks what songs will be played: "will they play X?", "what's on the setlist?". DO NOT use file_search for this, just call the check_setlist
   - Also call this proactively after analyze_concerts_for_user if user mentions favourite songs

3. find_similar_artists_touring
   - Call this ONLY if you DO NOT have the orginal artist in the file_search AND one of the following cases happens:
    - User wants alternatives: "who else is touring?", "any similar artists?"
    - Also use when the main artist has no concerts

4. get_flight_price
   - User asks for more flight options or flights to a specific city/date
   - NEVER invent flight prices - always call this tool

5. file_search
   - Background info only: genres, similar artists, descriptions
   - file_search does NOT have concert dates always use tools above for live data

PRESENTING RESULTS:
- Always show ALL concerts in concerts_analyzed, ranked by cheapest round-trip first
- For each concert ALWAYS show: artist, venue, city, date, ticket link, one-way price,
  round-trip price, airline, flight duration, flight link
- If fallback_note is present, show it prominently before listing concerts
- Local shows (is_local=True) always go first with "No flight needed!"
- NEVER invent or estimate prices — only show what the tools return

SETLIST RULES:
- If user mentions favourite songs, call check_setlist with those songs
- Clearly state which songs WILL be played and which will NOT
- Never guess - only report what check_setlist returns

CONVERSATION:
- Answer follow-up questions from results already in the conversation - no tool call needed
- If no origin city given, ask for it before calling analyze_concerts_for_user
- Be direct and honest like a friend who knows concerts"""

    tools = [
        {
            "type": "file_search",
            "vector_store_ids": [VECTOR_STORE_ID],
            "max_num_results": 5
        },
        {
            "type": "function",
            "name": "analyze_concerts_for_user",
            "description": (
                "Find concerts + flights for an artist from a city. "
                "ONLY call for a NEW artist or NEW city. "
                "Results ranked by cheapest round-trip flight."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "artist_name": {"type": "string"},
                    "origin_city": {"type": "string"}
                },
                "required": ["artist_name", "origin_city"]
            }
        },
        {
            "type": "function",
            "name": "check_setlist",
            "description": (
                "Check what songs an artist will play. "
                "Call when user asks 'will they play X?' or mentions favourite songs."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "artist_name": {"type": "string"},
                    "songs_to_check": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "Specific songs to check against the setlist"
                    }
                },
                "required": ["artist_name"]
            }
        },
        {
            "type": "function",
            "name": "find_similar_artists_touring",
            "description": (
                "Find similar artists with upcoming concerts if the original artist IS NOT in the vectore store."
                "ONLY call this AFTER analyze_concerts_for_user has returned has_concerts: False. "
                "Never call this as a first response to an artist query."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "original_artist": {"type": "string"},
                    "origin_city": {"type": "string"}
                },
                "required": ["origin_city"]
            }
        },
        {
            "type": "function",
            "name": "get_flight_price",
            "description": (
                "Get real flight prices between two cities. "
                "Call when user asks for more flight options or flights to a specific city. "
                "NEVER invent prices — always call this."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "origin_city": {"type": "string"},
                    "destination_city": {"type": "string"},
                    "outbound_date": {"type": "string", "description": "YYYY-MM-DD concert date"},
                    "return_date": {"type": "string", "description": "YYYY-MM-DD optional"}
                },
                "required": ["origin_city", "destination_city", "outbound_date"]
            }
        }
    ]


    previous_response_id = None
    session_city = None

    while True:
        user_input = input("\n You: ").strip()
        if user_input.lower() in ['quit', 'exit', 'bye']:
            print("\n Hope I helped!")
            break
        if not user_input:
            continue

        try:
            payload = (
                [{"role": "system", "content": SYSTEM_PROMPT},
                 {"role": "user",   "content": user_input}]
                if previous_response_id is None
                else user_input
            )

            for city in AIRPORT_CODES.keys():
              if city in user_input.lower():
                  session_city = city
                  # print(f"City stored: {session_city}")
                  break

            #innject stored city into the message if not already there
            if session_city and session_city not in user_input.lower():
                user_message = f"{user_input}\n[User's city from earlier in conversation: {session_city}]"
            else:
              user_message = user_input

            if session_city and looks_like_concert_query(user_input):
                tool_choice = {"type": "function", "name": "analyze_concerts_for_user"}
            elif previous_response_id is None:
                tool_choice = "required"
            else:
                tool_choice = "auto"

            response = client.responses.create(
                model="gpt-4o-mini",
                input=payload,
                tools=tools,
                tool_choice=tool_choice,
                previous_response_id=previous_response_id,
            )

            while response.status == "requires_action" or any(
                getattr(item, "type", None) == "function_call"
                for item in (response.output or [])
            ):
                tool_results = []

                for item in response.output:
                    if getattr(item, "type", None) != "function_call":
                        continue

                    fn_name = item.name

                    try:
                        fn_args = json.loads(item.arguments)
                    except (json.JSONDecodeError, AttributeError):
                        fn_args = {}

                    if fn_name == "analyze_concerts_for_user":
                        result = analyze_concerts_for_user(**fn_args)
                    elif fn_name == "check_setlist":
                        result = check_setlist(**fn_args)
                    elif fn_name == "find_similar_artists_touring":
                        result = find_similar_artists_touring(**fn_args)
                    elif fn_name == "get_flight_price":
                        result = get_flight_price(**fn_args)
                    else:
                        result = {"error": f"Unknown function: {fn_name}"}

                    tool_results.append({
                        "type": "function_call_output",
                        "call_id": item.call_id,
                        "output": json.dumps(result)
                    })

                if not tool_results:
                    break

                response = client.responses.create(
                    model="gpt-4o-mini",
                    input=tool_results,
                    tools=tools,
                    previous_response_id=response.id,
                )

            previous_response_id = response.id
            text = response.output_text
            if text:
                print(f"\n Assistant: {text}")
            else:
                print("\n Assistant: Let me check on that for you...")

        except KeyboardInterrupt:
            print("\n\n Goodbye!")
            break
        except Exception as e:
            print(f"\n Error: {e}")
            print("   Please try again.")




In [ ]:
run_concert_travel_agent()